# 🌍 ISOM 260: Agents Meet the Real World

**Session 5 — From Simulated to Real** | Suffolk University | Prof. Hasan Arslan

---

## From Session 4 to Session 5

In Session 4, you built your first AI agent. It had:
- 🧠 A **reasoning loop** — the agent decides what to do next
- 🛠️ **Tools** — a calculator, stock lookup, company info
- 🔄 The **agent loop** — send to LLM → call tool → send result back → repeat

But everything was **simulated**. The stock prices never changed. The company info was hardcoded. If you asked the same question twice, you got the same answer. And the agent had **no memory** — every conversation started from scratch.

### The Problem with Simulated Tools

```
Session 4 Agent:  "AAPL is at $242.58"  ← Same answer every time.
Real World:       "AAPL is at $187.32"  ← Changes every second.
```

### Today: Real APIs, Real Errors, Real Memory

In this session, we upgrade our agent with:

1. 📡 **Real APIs** — Wikipedia search, web page fetcher (live data from the internet!)
2. ⚠️ **Error handling** — what happens when an API fails? (spoiler: real APIs fail a lot)
3. 🧠 **Conversation memory** — the agent remembers what you said 3 turns ago

### The Progression

```
Session 4  →  Session 5  →  Session 8
Simulated     Real APIs     Production Systems
tools         + errors      + ReAct reasoning
              + memory      + multi-agent
                            + MCP protocol
```

### 💡 NanoClaw Context

Remember NanoClaw — the 500-line agent that went viral? It uses real APIs (email, calendar, web), handles errors gracefully, and maintains conversation memory across sessions. Today you'll build the same capabilities, one piece at a time.

---

## 🚀 Part 1: Setup (2 minutes)

Install dependencies and connect to Claude.

In [ ]:
# Install the Anthropic Python SDK + web scraping tools
!pip install -q anthropic beautifulsoup4 requests

In [ ]:
# Set your API key
# Get yours at: https://console.anthropic.com
import os
from google.colab import userdata

# Option 1: Use Colab Secrets (recommended)
# Go to the key icon in the left sidebar → Add secret named ANTHROPIC_API_KEY
try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("✅ API key loaded from Colab Secrets!")
except:
    # Option 2: Paste directly (less secure, but works for class)
    os.environ["ANTHROPIC_API_KEY"] = "your-api-key-here"  # <-- Replace this
    print("⚠️ Using hardcoded API key. Consider using Colab Secrets instead.")

In [ ]:
# Verify connection
import anthropic

client = anthropic.Anthropic()

response = client.messages.create(
    model="claude-sonnet-4-5-20250929",
    max_tokens=100,
    messages=[{"role": "user", "content": "Say 'Agent ready!' in exactly 2 words."}]
)

print(response.content[0].text)
print(f"\n✅ Connection successful! Using model: claude-sonnet-4-5-20250929")

---

## 🔧 Part 2: Bring Back the Calculator (Foundation from Session 4)

Before we add real-world tools, let's bring back our Session 4 foundation: the calculator tool and the agent loop. This is the **same code** from last session — we're building on top of it.

In [ ]:
# ============================================
# STEP 1: Define the tool for Claude
# ============================================
# This is like a job description — it tells Claude what the tool can do

calculator_tool = {
    "name": "calculator",
    "description": (
        "A precise mathematical calculator. Use this tool whenever you need to "
        "perform arithmetic calculations. It handles addition, subtraction, "
        "multiplication, division, and exponentiation with perfect accuracy. "
        "Always prefer this tool over mental math for any calculation."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "The mathematical expression to evaluate, e.g. '2 + 2' or '(100 * 0.15) + 50'"
            }
        },
        "required": ["expression"]
    }
}

print("📝 Tool defined: calculator")
print(f"   Description: {calculator_tool['description'][:80]}...")

In [ ]:
# ============================================
# STEP 2: Create the actual tool function
# ============================================
# This is the real Python code that runs when Claude calls the tool

def calculator(expression: str) -> str:
    """
    Safely evaluate a mathematical expression.
    Returns the result as a string.
    """
    try:
        # Only allow safe math operations
        allowed_chars = set('0123456789+-*/.() ')
        if not all(c in allowed_chars for c in expression):
            return f"Error: Expression contains invalid characters. Only numbers and +-*/.() are allowed."

        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"

# Test it!
print("Testing calculator:")
print(f"  2 + 2 = {calculator('2 + 2')}")
print(f"  7394 * 8261 = {calculator('7394 * 8261')}")
print(f"  (100 * 0.15) + 50 = {calculator('(100 * 0.15) + 50')}")
print("\n✅ Calculator tool is working!")

In [ ]:
# ============================================
# STEP 3: Build the Agent Loop!
# ============================================
# This is the HEART of every AI agent — the loop that connects
# Claude's thinking to real-world tool execution

import json

def run_agent(user_message: str, tools: list, tool_functions: dict, verbose=True):
    """
    Run an AI agent that can use tools to answer questions.

    This is the same core pattern used by NanoClaw, OpenClaw,
    Claude Code, and every AI agent in production today.
    """
    if verbose:
        print(f"\n{'='*60}")
        print(f"🗣️  User: {user_message}")
        print(f"{'='*60}")

    messages = [{"role": "user", "content": user_message}]
    step = 0

    while True:
        step += 1

        # 📡 Send message to Claude (with tool definitions)
        response = client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=1024,
            system="You are a helpful assistant. Use the available tools whenever they would help you give a more accurate answer. Always show your reasoning.",
            tools=tools,
            messages=messages
        )

        if verbose:
            print(f"\n🔄 Step {step} | Stop reason: {response.stop_reason}")

        # Check if Claude wants to use a tool
        if response.stop_reason == "tool_use":
            # Process ALL content blocks (Claude might think + use tools)
            tool_results = []

            for block in response.content:
                if block.type == "text":
                    if verbose:
                        print(f"   🧠 Thinking: {block.text[:150]}..." if len(block.text) > 150 else f"   🧠 Thinking: {block.text}")

                elif block.type == "tool_use":
                    tool_name = block.name
                    tool_input = block.input

                    if verbose:
                        print(f"   🛠️  Calling tool: {tool_name}")
                        print(f"      Input: {json.dumps(tool_input)}")

                    # 🚀 EXECUTE the tool!
                    if tool_name in tool_functions:
                        result = tool_functions[tool_name](**tool_input)
                    else:
                        result = f"Error: Unknown tool '{tool_name}'"

                    if verbose:
                        print(f"      Result: {result[:200]}..." if len(result) > 200 else f"      Result: {result}")

                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result
                    })

            # Add Claude's response and tool results to conversation
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": tool_results})

        else:
            # Claude is done — extract final answer
            final_answer = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_answer += block.text

            if verbose:
                print(f"\n🤖 Agent Answer:\n{final_answer}")
                print(f"\n✅ Done in {step} step(s)")

            return final_answer

        # Safety: prevent infinite loops
        if step > 10:
            return "Error: Agent exceeded maximum steps."

print("✅ Agent loop defined! Let's test it.")

In [ ]:
# ============================================
# 🎯 Quick Test — Verify Session 4 foundation works
# ============================================

# Register our tools
tools = [calculator_tool]
tool_functions = {"calculator": calculator}

# Test: Simple math
run_agent("What is 7,394 multiplied by 8,261?", tools, tool_functions)

---

## 📡 Part 3: Wikipedia Search Tool (REAL API!)

Here's where things get exciting. Instead of hardcoded data, we're going to call a **real API** — Wikipedia's REST API.

```
https://en.wikipedia.org/api/rest_v1/page/summary/{topic}
```

This is:
- 🆓 **Free** — no API key needed
- 📡 **Live** — returns real, up-to-date information
- 📦 **JSON** — returns structured data our agent can parse

This is the same kind of API that production agents like NanoClaw use to access real-world information.

In [ ]:
import requests

# ============================================
# 📡 Wikipedia Search Tool — REAL API!
# ============================================

# Tool definition
wikipedia_tool = {
    "name": "wikipedia_search",
    "description": (
        "Search Wikipedia for information about any topic. Returns the page title, "
        "a summary, and the URL. Use this when you need factual information about "
        "people, places, companies, concepts, or events. The topic should be "
        "formatted as a Wikipedia page title (e.g., 'Suffolk_University', 'Artificial_intelligence')."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "topic": {
                "type": "string",
                "description": "The topic to search for on Wikipedia, e.g. 'Artificial_intelligence' or 'Tesla,_Inc.'"
            }
        },
        "required": ["topic"]
    }
}

def wikipedia_search(topic: str) -> str:
    """Search Wikipedia for a topic — this is a REAL API call!"""
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{topic}"
    try:
        response = requests.get(url, timeout=10,
                               headers={"User-Agent": "ISOM260-Agent/1.0"})
        if response.status_code == 404:
            return json.dumps({"error": f"No Wikipedia page found for '{topic}'. Try a different search term or check spelling."})
        response.raise_for_status()
        data = response.json()
        return json.dumps({
            "title": data.get("title", ""),
            "summary": data.get("extract", ""),
            "url": data.get("content_urls", {}).get("desktop", {}).get("page", "")
        })
    except requests.exceptions.Timeout:
        return json.dumps({"error": "Request timed out. Wikipedia might be slow — try again."})
    except requests.exceptions.RequestException as e:
        return json.dumps({"error": f"Request failed: {str(e)}"})

print("✅ Wikipedia search tool defined — this one calls a REAL API!")

In [ ]:
# Test the raw function first (before giving it to the agent)
print("📡 Calling Wikipedia API directly...\n")

result = wikipedia_search("Suffolk_University")
data = json.loads(result)
print(f"Title: {data['title']}")
print(f"Summary: {data['summary'][:200]}...")
print(f"URL: {data['url']}")
print("\n🎉 That's LIVE data from Wikipedia — not simulated!")

In [ ]:
# Register calculator + Wikipedia
tools = [calculator_tool, wikipedia_tool]
tool_functions = {"calculator": calculator, "wikipedia_search": wikipedia_search}

run_agent("What is artificial intelligence? Give me a brief summary.", tools, tool_functions)

---

## 🌐 Part 4: Web Page Fetcher Tool (REAL API!)

Wikipedia is great, but agents need to read **any web page**. What if someone asks about a news article? A company's About page? A government report?

Enter `requests` + `BeautifulSoup` — the classic Python combo for web scraping. We'll build a tool that:
1. Fetches any URL
2. Strips out HTML, scripts, and navigation
3. Returns clean text the agent can read

In [ ]:
from bs4 import BeautifulSoup

# ============================================
# 🌐 Web Page Fetcher Tool — REAL API!
# ============================================

web_fetch_tool = {
    "name": "web_fetch",
    "description": (
        "Fetch and read the text content of any web page given its URL. "
        "Returns the extracted text content (up to 3000 characters). "
        "Use this when you need to read a specific web page, article, or document. "
        "Provide the full URL including https://."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "url": {
                "type": "string",
                "description": "The full URL to fetch, e.g. 'https://example.com'"
            }
        },
        "required": ["url"]
    }
}

def web_fetch(url: str) -> str:
    """Fetch a web page and extract its text content."""
    try:
        headers = {"User-Agent": "ISOM260-Agent/1.0"}
        response = requests.get(url, timeout=15, headers=headers)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        # Remove script and style elements
        for element in soup(["script", "style", "nav", "footer", "header"]):
            element.decompose()

        text = soup.get_text(separator="\n", strip=True)

        # Truncate to 3000 characters
        if len(text) > 3000:
            text = text[:3000] + "\n\n[...truncated to 3000 characters]"

        return json.dumps({
            "url": url,
            "content": text,
            "length": len(text)
        })
    except requests.exceptions.Timeout:
        return json.dumps({"error": f"Timeout: '{url}' took too long to respond."})
    except requests.exceptions.ConnectionError:
        return json.dumps({"error": f"Connection error: Could not reach '{url}'. Check the URL."})
    except requests.exceptions.HTTPError as e:
        return json.dumps({"error": f"HTTP error {e.response.status_code} for '{url}'."})
    except Exception as e:
        return json.dumps({"error": f"Failed to fetch '{url}': {str(e)}"})

print("✅ Web fetcher tool defined — reads ANY web page!")

In [ ]:
# Register all 3 tools
all_tools = [calculator_tool, wikipedia_tool, web_fetch_tool]
all_functions = {
    "calculator": calculator,
    "wikipedia_search": wikipedia_search,
    "web_fetch": web_fetch
}
print("✅ 3 tools registered:")
for t in all_tools:
    print(f"   🛠️  {t['name']}")

In [ ]:
# 🎯 Multi-tool test: Wikipedia + Calculator working together

run_agent(
    "Look up Suffolk University on Wikipedia and tell me when it was founded. "
    "Then calculate how many years it has been operating as of 2026.",
    all_tools, all_functions
)

### 🌟 Key Insight: Real vs. Simulated

Notice the difference from Session 4:

| | Session 4 (Simulated) | Session 5 (Real) |
|---|---|---|
| Data source | Hardcoded Python dict | Live Wikipedia API |
| Freshness | Always the same | Updated in real-time |
| Coverage | 6 companies only | Every Wikipedia article |
| Network | No internet needed | Requires internet |
| Can fail? | Never | Yes (404, timeout, etc.) |

That last row — "Can fail?" — is the big one. Real APIs fail. A lot. Let's learn to handle that.

---

## ⚠️ Part 5: Error Handling — When the Real World Breaks

In Session 4, nothing ever failed. That's because simulated tools always return the same hardcoded data.

Real APIs? They fail constantly:
- 🔴 **404** — page doesn't exist
- 🔴 **Timeout** — server too slow
- 🔴 **Connection error** — no internet / bad URL
- 🔴 **403** — forbidden / blocked
- 🔴 **500** — server crashed

Production agents don't crash on errors. They **handle them gracefully** and tell the user what happened. Let's test it.

In [ ]:
# ============================================
# ⚠️ Test 1: Wikipedia page that doesn't exist
# ============================================

run_agent(
    "Look up 'Xyzzy_Nonexistent_Page_12345' on Wikipedia.",
    all_tools, all_functions
)

In [ ]:
# ============================================
# ⚠️ Test 2: URL that doesn't exist
# ============================================

run_agent(
    "Fetch the web page at https://thiswebsitedoesnotexist12345.com",
    all_tools, all_functions
)

In [ ]:
# ============================================
# 🧪 YOUR TURN: Test 3 Error Scenarios
# ============================================
# Try these (or come up with your own!):
#
# 1. A Wikipedia topic with unusual characters
# 2. A URL that returns a 403 (forbidden)
# 3. A very long Wikipedia topic name
#
# What happens? Does the agent crash or adapt?

run_agent(
    "YOUR ERROR TEST HERE",  # <-- Replace with your test
    all_tools, all_functions
)

### 🌟 Key Insight: Graceful Failure

Notice: the agent **doesn't crash**. It gets the error message, understands it, and gives you a helpful response. This is what production agents do.

The pattern:
```
1. Tool function catches the exception
2. Returns a JSON error message (NOT a Python crash)
3. Claude reads the error and explains it to the user
4. Agent continues running — ready for the next question
```

In NanoClaw and production systems, this is taken further with **retries** (try 3 times before giving up), **fallbacks** (if Wikipedia fails, try a web search), and **alerts** (notify the developer if errors spike).

---

## 🧠 Part 6: Agent Memory — Remembering Across Turns

Here's a problem you might not have noticed yet. Our agent has **no memory**.

Every time you call `run_agent()`, it starts a brand-new conversation. It has no idea what you asked before. Watch:

In [ ]:
# ============================================
# 🧠 THE PROBLEM: No Memory!
# ============================================
# Watch what happens with two related questions:

print("Question 1:")
run_agent("Who founded Tesla Motors?", all_tools, all_functions)

print("\n" + "="*60)
print("Question 2 (follow-up):")
run_agent("How old is he now in 2026?", all_tools, all_functions)
# 💥 The agent has NO IDEA who "he" refers to!

In [ ]:
# ============================================
# 🧠 THE FIX: Agent with Memory!
# ============================================

def run_agent_with_memory(user_message: str, tools: list, tool_functions: dict,
                          conversation_history: list = None, verbose=True):
    """
    Agent with memory! Maintains conversation history across turns.

    The secret? conversation_history is just a list of messages.
    We send the FULL list to Claude every time. That's it. That's memory.
    """
    if conversation_history is None:
        conversation_history = []

    if verbose:
        print(f"\n{'='*60}")
        print(f"🗣️  User: {user_message}")
        print(f"   📝 Memory: {len(conversation_history)} previous messages")
        print(f"{'='*60}")

    # Add user message to history
    conversation_history.append({"role": "user", "content": user_message})

    step = 0

    while True:
        step += 1

        # Send FULL conversation history (that's the memory!)
        response = client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=1024,
            system="You are a helpful assistant. Use the available tools whenever they would help you give a more accurate answer. Always show your reasoning. Remember previous messages in our conversation.",
            tools=tools,
            messages=conversation_history
        )

        if verbose:
            print(f"\n🔄 Step {step} | Stop reason: {response.stop_reason}")

        if response.stop_reason == "tool_use":
            tool_results = []

            for block in response.content:
                if block.type == "text":
                    if verbose:
                        print(f"   🧠 Thinking: {block.text[:150]}..." if len(block.text) > 150 else f"   🧠 Thinking: {block.text}")
                elif block.type == "tool_use":
                    tool_name = block.name
                    tool_input = block.input

                    if verbose:
                        print(f"   🛠️  Calling tool: {tool_name}")
                        print(f"      Input: {json.dumps(tool_input)}")

                    if tool_name in tool_functions:
                        result = tool_functions[tool_name](**tool_input)
                    else:
                        result = f"Error: Unknown tool '{tool_name}'"

                    if verbose:
                        print(f"      Result: {result[:200]}..." if len(result) > 200 else f"      Result: {result}")

                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result
                    })

            # Add to conversation history
            conversation_history.append({"role": "assistant", "content": response.content})
            conversation_history.append({"role": "user", "content": tool_results})

        else:
            final_answer = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_answer += block.text

            if verbose:
                print(f"\n🤖 Agent Answer:\n{final_answer}")
                print(f"\n✅ Done in {step} step(s) | Memory: {len(conversation_history)} messages")

            # Add assistant response to history
            conversation_history.append({"role": "assistant", "content": response.content})

            return final_answer, conversation_history

        if step > 10:
            return "Error: Agent exceeded maximum steps.", conversation_history

print("✅ Agent with memory defined!")

In [ ]:
# ============================================
# 🎯 Test Memory: Multi-Turn Conversation
# ============================================

history = []

# Turn 1: Research
answer, history = run_agent_with_memory(
    "Who founded Tesla Motors?",
    all_tools, all_functions, history
)

# Turn 2: Follow-up (uses memory!)
answer, history = run_agent_with_memory(
    "How old is he now in 2026?",
    all_tools, all_functions, history
)

# Turn 3: Calculation based on context
answer, history = run_agent_with_memory(
    "If he started the company at that age, how many years has he been running it?",
    all_tools, all_functions, history
)

In [ ]:
# ============================================
# 🔍 Peek Inside the Memory
# ============================================
# Memory is just a list! Let's look at it.

print(f"📝 Total messages in memory: {len(history)}\n")
for i, msg in enumerate(history):
    role = msg["role"]
    if isinstance(msg["content"], str):
        preview = msg["content"][:80]
    elif isinstance(msg["content"], list):
        preview = f"[{len(msg['content'])} items]"
    else:
        preview = str(msg["content"])[:80]
    print(f"   {i+1}. [{role:>9}] {preview}...")

print(f"\n💡 That's it! Memory = a Python list. Nothing magical.")

### 🌟 Key Insight: How Memory Actually Works

Memory is just a list of messages. Every LLM-based agent — from NanoClaw to ChatGPT — works this way. The "memory" is sending the full conversation history with each request. Simple, effective, and now you understand it.

```python
# This IS memory:
conversation_history = [
    {"role": "user", "content": "Who founded Tesla?"},
    {"role": "assistant", "content": "Elon Musk co-founded Tesla..."},
    {"role": "user", "content": "How old is he?"},
    # ... Claude sees ALL of this every time
]
```

In production, this list gets stored in a **database** so it persists across sessions. But the mechanism is identical.

---

## 🏆 Part 7: Challenge + Reflection

In [ ]:
# ============================================
# 🏆 CHALLENGE: Complex Multi-Tool + Memory Query
# ============================================

history = []

answer, history = run_agent_with_memory(
    "I'm researching universities in Boston. Look up Suffolk University on Wikipedia "
    "and tell me about it.",
    all_tools, all_functions, history
)

answer, history = run_agent_with_memory(
    "Now look up Harvard University. How much older is Harvard than Suffolk?",
    all_tools, all_functions, history
)

answer, history = run_agent_with_memory(
    "Based on what you've learned about both universities, which one might be better "
    "for someone interested in business and technology? Use the information from our conversation.",
    all_tools, all_functions, history
)

In [ ]:
# ============================================
# 🎨 YOUR TURN: Build a Real API Tool!
# ============================================
# Ideas for free, no-key APIs:
#   🌦️ Open-Meteo Weather: https://api.open-meteo.com/v1/forecast?latitude=42.36&longitude=-71.06&current_weather=true
#   📰 Hacker News: https://hacker-news.firebaseio.com/v0/topstories.json
#   🎲 Random Facts: https://uselessfacts.jsph.pl/api/v2/facts/random
#   🌍 Country Info: https://restcountries.com/v3.1/name/{country}
#
# Template:

my_tool = {
    "name": "your_tool_name",
    "description": "What does it do? Be specific!",
    "input_schema": {
        "type": "object",
        "properties": {
            "param1": {
                "type": "string",
                "description": "What this parameter means"
            }
        },
        "required": ["param1"]
    }
}

def your_tool_name(param1: str) -> str:
    try:
        response = requests.get(f"YOUR_API_URL/{param1}", timeout=10)
        response.raise_for_status()
        data = response.json()
        return json.dumps(data)
    except Exception as e:
        return json.dumps({"error": str(e)})

# Register and test
# my_tools = all_tools + [my_tool]
# my_functions = {**all_functions, "your_tool_name": your_tool_name}
# run_agent("Test question", my_tools, my_functions)

---

## 💭 Reflection — Session 4 vs. Session 5 vs. Production

| | Session 4 | Session 5 | Production (NanoClaw) |
|---|---|---|---|
| Tools | Simulated (fake data) | Real APIs (Wikipedia, web) | Hundreds of real tools |
| Errors | Never fail | Handled gracefully | Retry, fallback, alert |
| Memory | None | Conversation history | Database-backed |
| Data | Always the same | Live from the internet | Real-time, multi-source |
| Agent loop | ~50 lines | ~60 lines | ~500 lines |

The **core pattern is identical**. The difference is reliability, scale, and real data.

```python
# Session 4, Session 5, and NanoClaw all do this:
while True:
    response = llm.call(messages, tools)
    if response.wants_tool:
        result = execute_tool(response.tool_call)
        messages.append(result)
    else:
        return response.answer
```

---

## 🚀 What's Next?

- **Session 6**: Computer Vision — teaching AI to see and understand images
- **Session 8**: AI Agents & Automation — ReAct reasoning, multi-agent systems, MCP protocol
- **Homework**: Add a real API tool (weather, news, or your own!), test edge cases, and design a memory strategy

---

**ISOM 260: AI for Business** | Suffolk University | Session 5

🌐 [isom-260.vercel.app](https://isom-260.vercel.app)